# Modul 6: Integrasi Teknik AI dalam Sistem Informasi
## Studi Kasus 1: Sistem Informasi Perpustakaan (Prediksi Risiko Keterlambatan Pengembalian Buku)

### Deskripsi Studi Kasus & Arsitektur Komponen
Notebook ini mengimplementasikan arsitektur berlapis (*Layered Architecture*) untuk memprediksi risiko keterlambatan pengembalian buku (`0` = Tepat Waktu, `1` = Berisiko Terlambat) berdasarkan riwayat peminjaman anggota:
1. **Model Layer (Fase Training & Serialisasi):** Melatih `MLPClassifier` menggunakan `scikit-learn` dan menyimpannya ke file `model_keterlambatan.joblib`.
2. **Data Layer (SQLite3):** Menginisialisasi dan mencatat setiap permintaan prediksi ke dalam tabel `log_peminjaman` pada `log_perpustakaan.db`.
3. **Application Layer (`templates/index.html`):** Membuat antarmuka web sederhana untuk berinteraksi dengan API.
4. **API Layer (Flask):** Menyediakan endpoint REST `/predict` (POST) untuk melayani inferensi model secara *real-time* (Fase Serving).
5. **Pengujian End-to-End:** Menguji API melalui HTTP POST menggunakan `requests` (dan `test_client` Flask agar bisa langsung dieksekusi di dalam satu notebook) serta memverifikasi isi database log.

### 1. Instalasi & Impor Pustaka
Pastikan pustaka berikut telah terpasang di lingkungan Python Anda.

In [ ]:
# Jalankan baris di bawah jika pustaka belum terpasang:
# !pip install scikit-learn joblib flask requests numpy

import os
import sqlite3
import threading
import time
from datetime import datetime
import joblib
import numpy as np
import requests
from flask import Flask, jsonify, render_template, request
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

### 2. Model Layer: Pelatihan & Serialisasi Model (`train_model.py`)
* **Fitur Input:** `[jumlah_buku, rata_lama_pinjam, jenis_anggota]` (`0` = Mahasiswa, `1` = Dosen/Staf, `2` = Umum)
* **Target Label:** `0` = Tepat Waktu, `1` = Berisiko Terlambat

In [ ]:
# Dataset historis peminjaman perpustakaan
# Fitur: [jumlah_buku_dipinjam, lama_rata_rata_peminjaman_hari, jenis_anggota]
X = np.array([
    [1, 3.0, 0],   # Mahasiswa, pinjam sedikit & cepat kembali -> Tepat waktu (0)
    [2, 4.5, 1],   # Dosen, kembali cepat -> Tepat waktu (0)
    [5, 12.0, 0],  # Mahasiswa, pinjam banyak & lama -> Terlambat (1)
    [4, 14.5, 2],  # Umum, pinjam banyak & lama -> Terlambat (1)
    [1, 5.0, 1],   # Dosen, pinjam 1 buku -> Tepat waktu (0)
    [6, 15.0, 0],  # Mahasiswa, beban bacaan tinggi & lama -> Terlambat (1)
    [2, 6.0, 2],   # Umum, cepat kembali -> Tepat waktu (0)
    [5, 11.5, 2],  # Umum, pinjam banyak & lama -> Terlambat (1)
    [3, 5.5, 0],   # Mahasiswa, sedang & tepat waktu -> Tepat waktu (0)
    [4, 13.0, 1],  # Dosen, pinjam banyak & sangat lama -> Terlambat (1)
    [2, 3.5, 0],   # Mahasiswa, cepat kembali -> Tepat waktu (0)
    [6, 16.0, 2],  # Umum, sangat berisiko terlambat -> Terlambat (1)
])

y = np.array([0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1])

# Membagi data latih dan data uji (75% latih, 25% uji)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Pipeline: StandardScaler + MLPClassifier
model = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(8, 4), activation="relu", max_iter=2000, random_state=42)
)

# Melatih model
model.fit(X_train, y_train)
print(f"Akurasi model pada data uji: {model.score(X_test, y_test):.2f}")

# Serialisasi model ke file fisik
nama_file_model = "model_keterlambatan.joblib"
joblib.dump(model, nama_file_model)
print(f"Model berhasil disimpan sebagai '{nama_file_model}'")

### 3. Application Layer: Membuat Antarmuka Web (`templates/index.html`)
Sel ini membuat folder `templates/` dan file `index.html` secara otomatis sehingga dapat dirender oleh Flask.

In [ ]:
os.makedirs("templates", exist_ok=True)

html_content = '''<!DOCTYPE html>
<html lang="id">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Prediksi Keterlambatan Buku - Perpustakaan</title>
    <style>
        body { font-family: sans-serif; max-width: 480px; margin: 40px auto; line-height: 1.6; }
        .form-group { margin-bottom: 14px; }
        label { display: block; font-weight: bold; margin-bottom: 4px; }
        input, select, button { width: 100%; padding: 8px; box-sizing: border-box; }
        button { background: #111; color: #fff; border: none; cursor: pointer; font-weight: bold; margin-top: 8px; }
        button:hover { background: #333; }
        #hasil { margin-top: 20px; padding: 12px; border: 1px solid #ccc; display: none; }
    </style>
</head>
<body>
    <h2>Prediksi Keterlambatan Pengembalian Buku</h2>
    <p>Masukkan data riwayat peminjaman anggota untuk memprediksi risiko keterlambatan.</p>

    <div class="form-group">
        <label for="jumlah_buku">Jumlah Buku Dipinjam:</label>
        <input type="number" id="jumlah_buku" min="1" placeholder="Contoh: 3" required>
    </div>

    <div class="form-group">
        <label for="rata_lama_pinjam">Lama Rata-Rata Peminjaman Sebelumnya (Hari):</label>
        <input type="number" id="rata_lama_pinjam" step="0.1" min="0" placeholder="Contoh: 6.5" required>
    </div>

    <div class="form-group">
        <label for="jenis_anggota">Jenis Anggota:</label>
        <select id="jenis_anggota">
            <option value="0">0 - Mahasiswa</option>
            <option value="1">1 - Dosen / Staf</option>
            <option value="2">2 - Anggota Umum</option>
        </select>
    </div>

    <button onclick="cekPrediksi()">Prediksi Risiko</button>

    <div id="hasil"></div>

    <script>
        async function cekPrediksi() {
            const hasilDiv = document.getElementById("hasil");
            const payload = {
                jumlah_buku: parseInt(document.getElementById("jumlah_buku").value),
                rata_lama_pinjam: parseFloat(document.getElementById("rata_lama_pinjam").value),
                jenis_anggota: parseInt(document.getElementById("jenis_anggota").value)
            };

            try {
                const response = await fetch("/predict", {
                    method: "POST",
                    headers: { "Content-Type": "application/json" },
                    body: JSON.stringify(payload)
                });

                const data = await response.json();
                hasilDiv.style.display = "block";

                if (response.ok) {
                    hasilDiv.innerHTML = `
                        <strong>Status Prediksi:</strong> ${data.label}<br>
                        <strong>Tingkat Keyakinan (Probabilitas):</strong> ${(data.probabilitas * 100).toFixed(1)}%
                    `;
                } else {
                    hasilDiv.innerHTML = `<strong>Error:</strong> ${data.error}`;
                }
            } catch (err) {
                hasilDiv.style.display = "block";
                hasilDiv.innerHTML = `<strong>Gagal terhubung ke API:</strong> ${err.message}`;
            }
        }
    </script>
</body>
</html>'''

with open("templates/index.html", "w", encoding="utf-8") as f:
    f.write(html_content)

print("File 'templates/index.html' berhasil dibuat.")

### 4. API Layer & Data Layer (`app.py`)
Mengonfigurasi basis data SQLite (`log_perpustakaan.db`), memuat model terlatih (`model_keterlambatan.joblib`), dan mendefinisikan endpoint Flask `/` serta `/predict`.

In [ ]:
app = Flask(__name__)
DB_NAME = "log_perpustakaan.db"

def init_db():
    conn = sqlite3.connect(DB_NAME)
    conn.execute('''
        CREATE TABLE IF NOT EXISTS log_peminjaman (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            waktu TEXT NOT NULL,
            bulan_tahun TEXT NOT NULL,
            jumlah_buku INTEGER NOT NULL,
            rata_lama_pinjam REAL NOT NULL,
            jenis_anggota INTEGER NOT NULL,
            prediksi INTEGER NOT NULL,
            label TEXT NOT NULL,
            probabilitas REAL NOT NULL
        )
    ''')
    conn.commit()
    conn.close()

def log_prediction(data, prediksi, label, probabilitas):
    sekarang = datetime.now()
    conn = sqlite3.connect(DB_NAME)
    conn.execute(
        '''
        INSERT INTO log_peminjaman (
            waktu, bulan_tahun, jumlah_buku, rata_lama_pinjam,
            jenis_anggota, prediksi, label, probabilitas
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        ''',
        (
            sekarang.isoformat(),
            sekarang.strftime("%Y-%m"),
            data["jumlah_buku"],
            data["rata_lama_pinjam"],
            data["jenis_anggota"],
            prediksi,
            label,
            probabilitas,
        ),
    )
    conn.commit()
    conn.close()

init_db()
loaded_model = joblib.load("model_keterlambatan.joblib")

@app.route("/")
def home():
    return render_template("index.html")

@app.route("/predict", methods=["POST"])
def predict():
    data = request.get_json()
    if not data:
        return jsonify({"error": "Payload JSON tidak ditemukan"}), 400

    try:
        jumlah_buku = int(data["jumlah_buku"])
        rata_lama_pinjam = float(data["rata_lama_pinjam"])
        jenis_anggota = int(data["jenis_anggota"])

        if jenis_anggota not in [0, 1, 2]:
            return jsonify({"error": "jenis_anggota harus 0 (Mahasiswa), 1 (Dosen), atau 2 (Umum)"}), 400

        fitur = np.array([[jumlah_buku, rata_lama_pinjam, jenis_anggota]])
    except KeyError as e:
        return jsonify({"error": f"Field wajib hilang: {e}"}), 400
    except (ValueError, TypeError):
        return jsonify({"error": "Tipe data input tidak valid"}), 400

    prediksi = int(loaded_model.predict(fitur)[0])
    probabilitas = float(loaded_model.predict_proba(fitur)[0][prediksi])
    label = "Berisiko Terlambat" if prediksi == 1 else "Tepat Waktu"

    log_prediction(
        {
            "jumlah_buku": jumlah_buku,
            "rata_lama_pinjam": rata_lama_pinjam,
            "jenis_anggota": jenis_anggota,
        },
        prediksi,
        label,
        round(probabilitas, 3),
    )

    return jsonify({
        "prediksi": prediksi,
        "label": label,
        "probabilitas": round(probabilitas, 3),
    })

print("Aplikasi Flask & SQLite siap.")

### 5. Menjalankan Server & Pengujian End-to-End (`test_api.py`)
Agar dapat dieksekusi langsung di dalam Jupyter Notebook tanpa memblokir sel (*blocking*), server Flask dijalankan pada *background thread* pada port `5001`, lalu diuji menggunakan `requests.post()` dan diverifikasi langsung ke database SQLite.

In [ ]:
PORT = 5001

def run_server():
    app.run(port=PORT, debug=False, use_reloader=False)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(1.5)  # Menunggu server siap

URL_API = f"http://127.0.0.1:{PORT}/predict"

test_cases = [
    {"jumlah_buku": 1, "rata_lama_pinjam": 3.0, "jenis_anggota": 0},   # Mahasiswa rajin -> Tepat Waktu
    {"jumlah_buku": 6, "rata_lama_pinjam": 14.5, "jenis_anggota": 0},  # Mahasiswa pinjam banyak & lama -> Berisiko Terlambat
    {"jumlah_buku": 2, "rata_lama_pinjam": 5.0, "jenis_anggota": 1},   # Dosen -> Tepat Waktu
    {"jumlah_buku": 5, "rata_lama_pinjam": 13.0, "jenis_anggota": 2},  # Anggota Umum -> Berisiko Terlambat
]

print("=== 1. MENGUJI ENDPOINT REST API ===")
for idx, kasus in enumerate(test_cases, start=1):
    res = requests.post(URL_API, json=kasus)
    print(f"Kasus {idx}: Input={kasus} -> Status={res.status_code}, Output={res.json()}")

print("
=== 2. VERIFIKASI DATA LAYER (SQLITE LOG) ===")
conn = sqlite3.connect(DB_NAME)
cursor = conn.cursor()
cursor.execute("SELECT id, waktu, bulan_tahun, jumlah_buku, rata_lama_pinjam, jenis_anggota, label, probabilitas FROM log_peminjaman")
rows = cursor.fetchall()
conn.close()

print(f"Total baris tercatat di database: {len(rows)}")
for row in rows[-len(test_cases):]:
    print(row)

### 6. Jawaban Pertanyaan Latihan Mandiri

1. **Fitur Input Model & Relevansinya:**
   * `jumlah_buku`: Semakin banyak buku yang dipinjam sekaligus, semakin besar beban bacaan anggota sehingga peluang melewati tenggat pengembalian meningkat.
   * `rata_lama_pinjam`: Merepresentasikan kebiasaan historis kedisiplinan anggota dalam mengembalikan buku.
   * `jenis_anggota` (`0` = Mahasiswa, `1` = Dosen/Staf, `2` = Umum): Setiap kategori memiliki kebijakan batas waktu pinjam dan pola aktivitas akademik yang berbeda.
2. **Perancangan Skema Tabel Log untuk Analisis Lanjutan:**
   * Tabel `log_peminjaman` menyertakan kolom `waktu` (ISO-8601) dan kolom turunan `bulan_tahun` (`YYYY-MM`) untuk mempermudah kueri agregasi bulanan (`GROUP BY bulan_tahun, label`), serta menyimpan nilai fitur mentah dan `probabilitas` untuk mendeteksi pergeseran distribusi data (*data drift*).
3. **Risiko Jika API Model Tidak Tersedia (*Down*) & Solusinya:**
   * **Risiko:** Transaksi peminjaman di meja sirkulasi gagal/terhenti (*timeout*) jika pemanggilan API dilakukan secara sinkron tanpa penanganan error.
   * **Mitigasi:** Menerapkan *graceful degradation* (fallback ke aturan statis sederhana jika koneksi gagal/timeout), pemrosesan asinkron di latar belakang, serta menjalankan server di bawah *process manager* (`systemd` / Docker) dengan fitur *auto-restart*.